# Parse C.H.U.D. Results - Baselines and Stress-Testing Data

Parses the `.csv` data generated by C.H.U.D's baseline and ASR evaluations. 

Creates graphs for each of the three comparisons tested by C.H.U.D, with our interpretations.

In [5]:
from dataclasses import dataclass
import os
import re
import pandas as pd
import matplotlib.pyplot as plt

In [4]:
DATA_DIR = "./output/"
CREATE_MARKDOWN_TABLES = True
CREATE_GRAPHS          = True


ASR_RESULTS_PATTERN   = re.compile(r"""
                            ASR-
                            (?P<base>Chat|LoX)- # base model: "Chat" or "LoX"
                            (?P<n_gsm>\d+)-     # number of gsm samples
                            (?P<n_bt>\d+)       # number of beavertails samples
                            .csv""",
                        re.X)
                        
JUDGE_RESULTS_PATTERN = re.compile(r"""
                            judge-baseline-
                            (?P<dataset>advbench|beavertails) # dataset used in evaluation ("advbench" or "beavertails")
                            .csv""",
                        re.X)

In [125]:
# ===== Parse Judge Baseline Results =====

# 1. Open data files directory
baseline_data = [
    (os.path.join(DATA_DIR, path), match.group("dataset"))
    for path in os.listdir(DATA_DIR)
    if (match := re.match(JUDGE_RESULTS_PATTERN, path))
]

if baseline_data:
    print("Parsing Data:")
    for path, dataset in baseline_data:
        print(f"\t-{dataset:<11} baseline at {path}")
else:
    print("No baseline data files found.")

cols = ['Test Set', 'Samples', 'False Negative Rate', 'False Positive Rate', 'Overall Accuracy', 'Output Data']
header  = '| ' + ' | '.join(cols) + " |\n"
header += "| " + ' | '.join('-' * (len(col)-1) + ":" for col in cols) + " |\n"
table = header

for path, dataset in baseline_data:
    print(f"\n{'='*10} {dataset.upper():^11} Baseline {'='*10}\n")
    print(f"{path}\n")

    data = pd.read_csv(path)
    assert {'expected', 'correct'} <= set(data.columns), "CSV does not contain the expected columns..."
    expected = data['expected']
    correct  = data['correct']

    total_samples = len(data)
    total_safe     = expected[expected == 0].count()
    total_unsafe   = expected[expected == 1].count()
    total_correct  = correct[correct == True].count()
    false_positive = expected[(expected == 0) & (correct == 0)].count()
    false_negative = expected[(expected == 1) & (correct == 0)].count()

    correct_rate        = 100 * (total_correct / total_samples)
    false_positive_rate = 100 * (false_positive / total_safe)
    false_negative_rate = 100 * (false_negative / total_unsafe)
    

    table += (
        f"| "
        f"{dataset:^11} | "
        f"{total_samples:^4} | "
        f"{false_negative_rate:^5.2f} % | "
        f"{false_positive_rate:^5.2f} % | "
        f"{correct_rate:^6.2f} % | "
        f"{path} |\n"
    )
    if not CREATE_MARKDOWN_TABLES:
        print(
            f"Total samples:       {total_samples:>5}",
            f"  - Safe samples:    {total_safe:>5}",
            f"  - Unsafe samples:  {total_unsafe:>5}\n",
            f"Total correct:       {total_correct:>5} ({correct_rate:>5.2f} %)",
            f"  - False Negatives: {false_negative:>5} ({false_negative_rate:>5.2f} %)",
            f"  - False Positives: {false_positive:>5} ({false_positive_rate:>5.2f} %)\n",
            sep="\n"
        )

if CREATE_MARKDOWN_TABLES:
    print("="*10 + " Results Summary " + "="*10)
    print("\n" + table)

Parsing Data:
	-beavertails baseline at ./output/judge-baseline-beavertails.csv
	-advbench    baseline at ./output/judge-baseline-advbench.csv

========== BEAVERTAILS Baseline ==========

./output/judge-baseline-beavertails.csv


==========  ADVBENCH   Baseline ==========

./output/judge-baseline-advbench.csv

========== Results Summary ==========

| Test Set | Samples | False Negative Rate | False Positive Rate | Overall Accuracy | Output Data |
| -------: | ------: | ------------------: | ------------------: | ---------------: | ----------: |
| beavertails | 1000 | 47.60 % | 4.00  % | 74.20  % | ./output/judge-baseline-beavertails.csv |
|  advbench   | 200  | 2.00  % | 0.00  % | 99.00  % | ./output/judge-baseline-advbench.csv |

